In [1]:
import pathlib, json

data = {}

In [25]:
from collections import Counter
from tqdm.auto import tqdm

INCLUDE_ROOM_TYPES = {"living_room", "bedroom", "dining_room", "office"}

# vid2room_vlm_analyses = {}
# for scene_path in tqdm(interesting_scenes):
#     room_name = scene_path.name
#     room_type = room_name.rsplit("_", 1)[0]
#     if room_type not in INCLUDE_ROOM_TYPES:
#         continue
#     vlm_analysis_path = scene_path / "vlm_analysis.json"
#     with open(vlm_analysis_path, "r") as f:
#         vlm_analysis = json.load(f)
#     vid2room_vlm_analyses[scene_path] = vlm_analysis


def load_segmentation(scene_path):
    room_name = scene_path.name
    room_type = room_name.rsplit("_", 1)[0]
    if room_type not in INCLUDE_ROOM_TYPES:
        return None
    segmentation_path = scene_path / "segmentation3d/objects_to_segmentation_maps.json"
    if not segmentation_path.exists():
        return None
    with open(segmentation_path, "r") as f:
        return [k for k in json.load(f)]

def load_vid2room():
    interesting_scenes = [pathlib.Path(p) for p in json.load(open("/cvgl2/u/cgokmen/BEHAVIOR-1K/slurm/interesting_scenes_full.json"))]

    segmentations = {}
    for scene_path in tqdm(interesting_scenes):
        segmentation = load_segmentation(scene_path)
        if segmentation is not None:
            segmentations[scene_path] = segmentation

    # How many categories are there?
    STRUCTURE_CATEGORIES = {"wall", "floor", "ceiling", "window", "door", "doorway", "stairs"}
    objects_by_category = Counter()
    for scene_path, segmentation in segmentations.items():
        for obj in segmentation:
            category = obj.rsplit("-", 2)[0]
            if category in STRUCTURE_CATEGORIES:
                continue
            objects_by_category[category] += 1

    print(objects_by_category.most_common())

    # Report histogram of object count per scene. Use plotly
    object_counts = [len(segmentation) for segmentation in segmentations.values()]

    return dict(objects_by_category=objects_by_category, scene_object_counts=object_counts)

data["vid2room"] = load_vid2room()

  0%|          | 0/37533 [00:00<?, ?it/s]

[('painting', 4781), ('pillow', 3400), ('dining_chair', 2309), ('rug', 1906), ('decorative_object', 1773), ('sofa', 1664), ('potted_plant', 1652), ('lamp', 1649), ('curtain', 1537), ('table_lamp', 1437), ('picture_frame', 1346), ('cabinet', 1332), ('cushion', 1318), ('armchair', 1242), ('side_table', 1047), ('vase_with_flowers', 974), ('bed', 947), ('curtains', 941), ('dresser', 907), ('coffee_table', 864), ('mirror', 820), ('nightstand', 807), ('book', 719), ('vase', 617), ('chandelier', 582), ('shelf', 579), ('desk', 545), ('ceiling_fan', 489), ('dining_table', 444), ('ottoman', 424), ('statue', 397), ('television', 381), ('floor_lamp', 374), ('chair', 365), ('tv', 352), ('bookcase', 340), ('blanket', 324), ('ceiling_light', 321), ('wall_sconce', 321), ('bar_stool', 316), ('bowl', 292), ('speaker', 250), ('wall_decoration', 239), ('console_table', 206), ('candle_holder', 205), ('basket', 200), ('pendant_light', 197), ('wall_decor', 195), ('framed_picture', 194), ('candle', 174), ('tv

In [ ]:
from collections import defaultdict

# Load the BEHAVIOR-1K versions of the same things
def load_behavior1k(dataset_name="behavior-1k-assets"):
    dataset_root = pathlib.Path("/scr2/datasets") / "behavior-1k-assets"
    scenes = sorted(dataset_root.glob("scenes/*/json/*_best.json"))

    unique_models_per_category = defaultdict(set)
    scene_object_counts = []

    for scene_json in tqdm(scenes):
        scene_data = json.loads(scene_json.read_text())
        room_object_counts = Counter()
        for obj in scene_data["objects_info"]["init_info"].values():
            category = obj["args"]["category"]
            unique_models_per_category[category].add(obj["args"]["model"])
            for room in obj["args"]["in_rooms"]:
                room_object_counts[room] += 1
        scene_object_counts.extend(room_object_counts.values())

    # Get the object counts by category
    objects_by_category = Counter()
    for category, models in unique_models_per_category.items():
        objects_by_category[category] = len(models)

    return dict(objects_by_category=objects_by_category, scene_object_counts=scene_object_counts)

data["behavior1k"] = load_behavior1k("behavior-1k-assets")
# data["hssd"] = load_behavior1k("hssd")
# data["spoc"] = load_behavior1k("spoc")

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

  0%|          | 0/51 [00:00<?, ?it/s]

In [33]:
import plotly.express as px
import pandas as pd

rows = []
for key, d in data.items():
    total_scene_count = sum(d["scene_object_counts"])
    for count in d["scene_object_counts"]:
        rows.append({"source": key, "object_count": count, "total_scene_count": total_scene_count})

df = pd.DataFrame(rows)

fig = px.histogram(df, x="object_count", color="source", nbins=100, range_x=[0, 200],
                   barmode="group", histnorm="probability density",
                   title="Object Count per Scene")
fig.show()

In [6]:
from transformers import AutoProcessor, AutoModel
model = AutoModel.from_pretrained("google/siglip-so400m-patch14-384").cuda()
processor = AutoProcessor.from_pretrained("google/siglip-so400m-patch14-384")

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [27]:
from tqdm.auto import trange
import torch
import numpy as np

all_unique_categories = sorted({c for d in data.values() for c in d["objects_by_category"]})

category_embeddings = []
for i in trange(0, len(all_unique_categories), 512, desc="Computing embeddings"):
    batch_texts = all_unique_categories[i:i+512]
    processed_texts = [x.replace("_", " ") for x in batch_texts]
    with torch.no_grad():
        inputs = processor(text=processed_texts, padding="max_length", return_tensors="pt").to(model.device)
        outputs = model.text_model(**inputs)
        text_embeds = outputs.pooler_output.cpu().numpy()
        category_embeddings.append(text_embeds)

embeddings = np.concatenate(category_embeddings, axis=0)
category_to_idx = {c: i for i, c in enumerate(all_unique_categories)}


Computing embeddings:   0%|          | 0/3 [00:00<?, ?it/s]

In [28]:
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import plotly.express as px

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
coords = tsne.fit_transform(embeddings)

colors = px.colors.qualitative.Plotly

fig = go.Figure()

for idx, (key, d) in enumerate(data.items()):
    cats = list(d["objects_by_category"].keys())
    cnts = np.array([d["objects_by_category"][c] for c in cats])
    indices = [category_to_idx[c] for c in cats]

    size_scale = np.sqrt(cnts)
    denom = size_scale.max() - size_scale.min()
    size_scale = 3 + 40 * (size_scale - size_scale.min()) / (denom if denom > 0 else 1)

    fig.add_trace(go.Scatter(
        x=coords[indices, 0], y=coords[indices, 1],
        mode="markers",
        marker=dict(
            size=size_scale,
            color=colors[idx % len(colors)],
            opacity=0.6,
            line=dict(width=0.5, color="white"),
        ),
        text=[f"{c} ({n})" for c, n in zip(cats, cnts)],
        hoverinfo="text",
        name=key,
    ))

    top_k = 10
    top_local = np.argsort(cnts)[-top_k:][::-1]
    top_global = [indices[i] for i in top_local]

    fig.add_trace(go.Scatter(
        x=coords[top_global, 0], y=coords[top_global, 1],
        mode="text",
        text=[cats[i].replace("_", " ") for i in top_local],
        textposition="top center",
        textfont=dict(size=11, color=colors[idx % len(colors)]),
        hoverinfo="skip",
        showlegend=False,
    ))

fig.update_layout(
    template="plotly_dark",
    title="t-SNE of Object Category Embeddings (SigLIP)",
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    showlegend=True,
    width=900,
    height=700,
)
fig.show()